In [1]:
# ############################################################
# Level 2 — 와인 데이터 + '가장 이상한 것' 콕 집기 (점수)
# ############################################################
# ------------------------------------------------------------
# [목적] 도구 불러오기 — 표 / 데이터 / 전처리 / 모델
# ------------------------------------------------------------
import pandas as pd                               # 표(DataFrame)로 확인하기
from sklearn.datasets import load_wine            # 와인 성분 13개 (연습용)
from sklearn.preprocessing import StandardScaler  # 스케일링 (단위 맞추기)
from sklearn.ensemble import IsolationForest      # 이상치 탐지

In [2]:
# ------------------------------------------------------------
# [데이터 살펴보기 · EDA] 와인 데이터는 어떻게 생겼나
#   · 와인 178병 × 성분 13개 (알코올·마그네슘 등, 단위가 제각각)
#   · 정답(종류)은 안 씀 = 비지도로 '유별난 와인'만 찾기
# ------------------------------------------------------------
X = load_wine(as_frame=True).data                 # 표 형태로 특성 13개만 받기
print('데이터 크기(행, 열):', X.shape)            # (178, 13)
X.head()                                           # 앞 5줄 미리보기 (성분 값들 보기)

데이터 크기(행, 열): (178, 13)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [3]:
# ------------------------------------------------------------
# [목적] 스케일링 후 판정 + '이상함의 정도(점수)'까지 표에 붙이기
#   (성분마다 값 크기가 달라 스케일링 필수)
# ------------------------------------------------------------
X_s = StandardScaler().fit_transform(X)           # 같은 잣대로 (평균0·표준편차1)
iso = IsolationForest(contamination=0.05, random_state=0)
X['anomaly'] = iso.fit_predict(X_s)               # 판정: -1=이상, 1=정상
X['score']   = iso.decision_function(X_s)         # 점수: 낮을수록 더 이상함
print('이상치 개수:', (X['anomaly'] == -1).sum(), '/', len(X))   # 개수 세기 (9개)

이상치 개수: 9 / 178


In [4]:
# ------------------------------------------------------------
# [목적] 점수 낮은 순 = 가장 이상한 와인부터 표로 보기
# ------------------------------------------------------------
X.sort_values('score').head()                     # 위로 올수록 '가장 유별난' 와인 (점수 낮음)

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,anomaly,score
121,11.56,2.05,3.23,28.5,119.0,3.18,5.08,0.47,1.87,6.00,0.93,3.69,465.0,-1,-0.088800
69,12.21,1.19,1.75,16.8,151.0,1.85,1.28,0.14,2.50,2.85,1.28,3.07,718.0,-1,-0.049909
59,12.37,0.94,1.36,10.6,88.0,1.98,0.57,0.28,0.42,1.95,1.05,1.82,520.0,-1,-0.046962
158,14.34,1.68,2.70,25.0,98.0,2.80,1.31,0.53,2.70,13.00,0.57,1.96,660.0,-1,-0.042046
110,11.46,3.74,1.82,19.5,107.0,3.18,2.58,0.24,3.58,2.90,0.75,2.81,562.0,-1,-0.022093


In [5]:
# ============================================================
# [결과 해석]
#  · 178개 중 9개를 이상치로 판정 (약 5%)
#  · decision_function 점수가 낮을수록 '더 유별남' -> 정렬로 가장 이상한 것부터 확인
#  · -1/1 판정만이 아니라 '순위'까지 알 수 있어 우선 살펴볼 대상을 고르기 좋음
# ============================================================